# Per-Category Model Evaluation — Safina Thapa

This notebook evaluates the **weighted Legal-BERT** model from Task 3 using the CUAD test dataset. It uses saved predictions, so we don't need to train the model again.

Since a contract chunk can belong to more than one category, this is a **multi-label classification** task. We'll calculate precision, recall, and F1 scores for all 41 categories, along with the overall micro and macro F1 scores.

Run the notebook from top to bottom using Python with pandas, NumPy, pyarrow, scikit-learn, and IPython installed. The results won't be saved to separate files unless the optional export step is enabled.

## 1. Model and threshold decision

We're using the weighted Legal-BERT model from Task 3. The training notebook (`Milestone2_Task3_Training_Optimization_Weighted_Model_Selection.ipynb`, sections 11–12) compared different prediction thresholds and kept 0.5 for the final model. We'll use the same threshold here, meaning probabilities of 0.5 or higher count as positive predictions.

The weighted model's prediction file is available locally, but its summary file isn't included. We'll calculate the evaluation scores directly from the saved predictions. Since Task 3 already compared thresholds using test data, these results aren't from a completely untouched test set.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.metrics import f1_score, precision_recall_fscore_support

THRESHOLD = 0.5
EXPECTED_ROWS = 2898
EXPECTED_CATEGORIES = 41
# A descriptive flag, not a statistical significance rule.
LOW_SUPPORT_LIMIT = 10
KEYS = ["title", "chunk_id"]

# Find the project whether the kernel starts in the root or notebooks folder.
working_directory = Path.cwd().resolve()
ROOT = next(
    (folder for folder in [working_directory, *working_directory.parents]
     if (folder / "data/processed/test_chunks_labeled.parquet").is_file()
     and (folder / "notebooks").is_dir()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Start the notebook from inside the project folder.")

PREDICTIONS_PATH = ROOT / "results/training/15ep/legal-bert-weighted/test_predictions.parquet"
LABELS_PATH = ROOT / "data/processed/test_chunks_labeled.parquet"
for path in [PREDICTIONS_PATH, LABELS_PATH]:
    if not path.is_file():
        raise FileNotFoundError(f"Required input is missing: {path}")

predictions = pd.read_parquet(PREDICTIONS_PATH)
actual = pd.read_parquet(LABELS_PATH)
print(f"Prediction rows: {len(predictions):,}; actual-label rows: {len(actual):,}")


Prediction rows: 2,898; actual-label rows: 2,898


## 2. Validate categories and match chunks

Before calculating the scores, we need to make sure the predictions and actual labels match correctly.

The labels file includes extra information, like contract text and character positions, that shouldn't be counted as categories. We'll check that all 41 categories are present, look for missing or duplicate chunk IDs, and make sure both files contain the same chunks.

We'll match the files using the contract title and chunk ID instead of assuming their rows are already in the same order.


In [2]:
ACTUAL_METADATA = KEYS + ["char_start", "char_end", "chunk_text"]

for name, frame in [("predictions", predictions), ("actual labels", actual)]:
    if not frame.columns.is_unique:
        raise ValueError(f"Duplicate column names in {name}.")
    if not set(KEYS).issubset(frame.columns):
        raise ValueError(f"Missing title or chunk_id in {name}.")
    if len(frame) != EXPECTED_ROWS:
        raise ValueError(f"Expected {EXPECTED_ROWS} rows in {name}, got {len(frame)}.")
    if frame[KEYS].isna().any().any():
        raise ValueError(f"Missing matching keys in {name}.")
    if frame.duplicated(KEYS).any():
        raise ValueError(f"Duplicate (title, chunk_id) keys in {name}.")

categories = [column for column in predictions.columns if column not in KEYS]
actual_categories = [column for column in actual.columns if column not in ACTUAL_METADATA]
if len(categories) != EXPECTED_CATEGORIES or len(actual_categories) != EXPECTED_CATEGORIES:
    raise ValueError("Both files must contain exactly 41 category columns.")
if set(categories) != set(actual_categories):
    raise ValueError(
        f"Category mismatch. Missing from labels: {set(categories) - set(actual_categories)}; "
        f"missing from predictions: {set(actual_categories) - set(categories)}"
    )

key_check = predictions[KEYS].merge(
    actual[KEYS], on=KEYS, how="outer", validate="one_to_one", indicator=True
)
if not key_check["_merge"].eq("both").all():
    raise ValueError(f"Unmatched chunks: {key_check['_merge'].value_counts().to_dict()}")

predictions_by_key = predictions.set_index(KEYS)
actual_aligned = actual.set_index(KEYS).loc[predictions_by_key.index, categories]
if not actual_aligned.index.equals(predictions_by_key.index):
    raise ValueError("Row alignment failed.")

# Validate BEFORE converting labels to integers, so bad values are not rounded away.
probabilities = predictions_by_key[categories].to_numpy(dtype=float)
label_values = actual_aligned.to_numpy(dtype=float)
if not np.isfinite(probabilities).all() or not ((probabilities >= 0) & (probabilities <= 1)).all():
    raise ValueError("Predictions must be finite probabilities between 0 and 1.")
if not np.isfinite(label_values).all() or not np.isin(label_values, [0, 1]).all():
    raise ValueError("Actual labels must contain only 0 and 1, with no missing values.")
if probabilities.shape != label_values.shape or probabilities.shape != (EXPECTED_ROWS, EXPECTED_CATEGORIES):
    raise ValueError("Unexpected evaluation matrix shape.")

y_true = label_values.astype(int)
y_pred = (probabilities >= THRESHOLD).astype(int)
print(f"Validated {len(actual_aligned):,} matched chunks and {len(categories)} categories.")
print(f"Positive prediction rule: probability >= {THRESHOLD}")


Validated 2,898 matched chunks and 41 categories.
Positive prediction rule: probability >= 0.5


## 3. Calculate and display metrics

We'll calculate the following metrics for each of the 41 clause categories:

- **Precision:** How many of the model's positive predictions were correct?
- **Recall:** How many of the actual positive examples did the model identify?
- **F1 score:** A score that combines precision and recall.
- **Support:** The number of test chunks that actually contain a category, not the number of unique contracts.

We'll also calculate two overall scores:

- **Micro F1:** Combines predictions across all categories before calculating F1.
- **Macro F1:** Takes the average F1 score across all 41 categories.

We use `zero_division=0` when a score can't be calculated because its denominator is zero. Categories with no positive test examples are still included in the results, with an F1 score of zero.

The training notebook excludes categories with no positive examples from some of its averages, so those scores may differ from the all-41 macro F1 reported here. Also, chunks from the same contract may overlap, so they aren't necessarily independent examples.


In [3]:
precision, recall, per_category_f1, support = precision_recall_fscore_support(
    y_true, y_pred, average=None, zero_division=0
)
results = pd.DataFrame({
    "category": categories,
    "precision": precision,
    "recall": recall,
    "f1": per_category_f1,
    "support": support.astype(int),
    "predicted_positives": y_pred.sum(axis=0),
})
results["limited_support"] = results["support"] < LOW_SUPPORT_LIMIT
results["no_test_positives"] = results["support"].eq(0)
results = results.sort_values(["f1", "category"], ascending=[False, True]).reset_index(drop=True)

micro_f1 = f1_score(y_true, y_pred, average="micro", zero_division=0)
macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
if not np.isclose(macro_f1, results["f1"].mean()):
    raise ValueError("Macro F1 does not match the mean category F1.")

overall = pd.DataFrame([{
    "model": "legal-bert-weighted",
    "threshold": THRESHOLD,
    "test_chunks": len(y_true),
    "categories": len(categories),
    "micro_f1": micro_f1,
    "macro_f1": macro_f1,
}])
# Round only the display; retain full precision in the data frames.
with pd.option_context("display.max_rows", 50, "display.max_columns", None):
    display(overall.round(4))
    display(results.round({"precision": 4, "recall": 4, "f1": 4}))


,model,threshold,test_chunks,categories,micro_f1,macro_f1
0,legal-bert-weighted,0.5,2898,41,0.7083,0.6416


,category,precision,recall,f1,support,predicted_positives,limited_support,no_test_positives
0,Source Code Escrow,1.0000,1.0000,1.0000,2,2,True,False
1,Governing Law,0.9091,0.9420,0.9253,138,143,False,False
2,Parties,0.8387,0.8844,0.8609,147,155,False,False
3,Cap On Liability,0.8534,0.8115,0.8319,122,116,False,False
4,No-Solicit Of Employees,0.7727,0.8947,0.8293,19,22,False,False
5,Insurance,0.7848,0.8732,0.8267,71,79,False,False
6,Expiration Date,0.7615,0.8919,0.8216,111,130,False,False
7,Anti-Assignment,0.7917,0.8382,0.8143,136,144,False,False
8,Audit Rights,0.7959,0.8298,0.8125,94,98,False,False
9,Renewal Term,0.6552,0.9048,0.7600,42,58,False,False


## 4. Limited support and interpretation

Some categories have very few positive examples in the test set, so their scores may not be very reliable. We'll flag categories with fewer than 10 positive chunks, including categories with none.

The cutoff of 10 is just a way to highlight categories with limited data. It doesn't mean that 10 examples are enough for a reliable evaluation.

The summary below is generated automatically from our results, so it will update if the evaluation scores change.


In [4]:
limited = results.loc[results["limited_support"]].sort_values(["support", "category"])
with pd.option_context("display.max_rows", 50):
    display(limited[["category", "support", "precision", "recall", "f1"]].round(4))

supported = results.loc[results["support"] > 0]
zero_names = results.loc[results["no_test_positives"], "category"].tolist()
interpretation = (
    f"At threshold **{THRESHOLD}**, weighted Legal-BERT has micro F1 **{micro_f1:.4f}** "
    f"and macro F1 **{macro_f1:.4f}** across all **{len(categories)} categories**. "
    f"**{len(limited)} categories** have fewer than {LOW_SUPPORT_LIMIT} positive test chunks. "
)
if not supported.empty:
    highest = supported.loc[supported["f1"].eq(supported["f1"].max()), "category"].tolist()
    lowest = supported.loc[supported["f1"].eq(supported["f1"].min()), "category"].tolist()
    interpretation += (
        f"Among categories with positive support, the highest F1 is {supported['f1'].max():.4f} "
        f"({'; '.join(highest)}), and the lowest is {supported['f1'].min():.4f} "
        f"({'; '.join(lowest)}). Read these scores together with their support. "
    )
if zero_names:
    interpretation += (
        f"No positive test chunks exist for: **{'; '.join(zero_names)}**. "
        "Their zero F1 values do not establish an inability to detect positive examples. "
    )
interpretation += "These are descriptive chunk-level results; they do not establish statistical significance or performance on unseen contract types."
display(Markdown(interpretation))


,category,support,precision,recall,f1
40,Price Restrictions,0,0.0000,0.0,0.0000
0,Source Code Escrow,2,1.0000,1.0,1.0000
39,Most Favored Nation,5,0.5000,0.2,0.2857
33,Unlimited/All-You-Can-Eat-License,5,0.6667,0.4,0.5000


At threshold **0.5**, weighted Legal-BERT has micro F1 **0.7083** and macro F1 **0.6416** across all **41 categories**. **4 categories** have fewer than 10 positive test chunks. Among categories with positive support, the highest F1 is 1.0000 (Source Code Escrow), and the lowest is 0.2857 (Most Favored Nation). Read these scores together with their support. No positive test chunks exist for: **Price Restrictions**. Their zero F1 values do not establish an inability to detect positive examples. These are descriptive chunk-level results; they do not establish statistical significance or performance on unseen contract types.

## 5. Save evaluation results

This step saves our evaluation results as two CSV files in `results/evaluation/`: one with the scores for all 41 categories and another with the overall micro and macro F1 scores.

The export is turned off by default. Set `EXPORT_RESULTS = True` to save the files. The code checks for existing files before saving so we don't accidentally overwrite previous results.

This step only saves evaluation results. It doesn't change the model or its training files.


In [7]:
EXPORT_RESULTS = False

if EXPORT_RESULTS:
    output_directory = ROOT / "results/evaluation"
    category_path = output_directory / "legal_bert_weighted_per_category.csv"
    overall_path = output_directory / "legal_bert_weighted_overall.csv"
    for path in [category_path, overall_path]:
        if path.exists():
            raise FileExistsError(f"Refusing to overwrite: {path}")
    output_directory.mkdir(parents=True, exist_ok=True)
    results.to_csv(category_path, index=False, mode="x")
    overall.to_csv(overall_path, index=False, mode="x")
    print(f"Saved {category_path.name} and {overall_path.name} in {output_directory}")
else:
    print("Export disabled. No output files were created.")


Export disabled. No output files were created.
